In [ ]:
from pathlib import Path
DATA_DIR = Path("../../data/analysis_ready")
MODEL_DIR = Path("../../outputs/full_models/30D_elec")
FIGURE_DIR = Path("../../outputs/figures/full/30D_elec")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
import pandas as pd
import numpy as np
elec_df = pd.read_csv(DATA_DIR / "elec_full_df.csv")
elec_y = pd.read_csv(DATA_DIR / "elec_full_y.csv")

                                        
elec_df.columns[elec_df.isna().any().tolist()]
elec_df['ClassificationOfAngina_PreOp_16'] = elec_df['ClassificationOfAngina_PreOp_16'].fillna("None")
cat_var = [x for x in elec_df.columns if x not in elec_df.columns[[43,46]]]
elec_df_cat = pd.get_dummies(elec_df[cat_var],columns=cat_var)

elec_df_num = elec_df.iloc[:,[43,46]]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
standardized  = scaler.fit_transform(elec_df_num)
elec_df_num = pd.DataFrame(standardized,columns=elec_df_num.columns)

df_new = pd.concat([elec_df_cat,elec_df_num],axis=1)
df_new['30Ddeath'] = elec_y['Deathwithin30DaysSinceOT_395']
df_new['30Ddeath']  = df_new['30Ddeath'] .map({'Y': 1, 'N': 0}).astype(int)


In [ ]:
from pycaret.classification import setup, compare_models, tune_model, blend_models, save_model, load_model, predict_model, get_config


In [ ]:
exp1 = setup(data=df_new, 
             target='30Ddeath',
             session_id=123,
             train_size=0.9999,
             preprocess=False,
             low_variance_threshold=0.1,
             remove_multicollinearity = True, 
             multicollinearity_threshold=0.9,
             fold=5,
             fold_strategy='stratifiedkfold',
             fold_shuffle=True,
             verbose=-1,
             n_jobs=20)


In [ ]:
best_model = compare_models(n_select=7,sort="AUC",exclude=['ridge','lightgbm','svm'])


In [ ]:
tuned_models = [tune_model(model,optimize="AUC",n_iter=50,choose_better=True) for model in best_model]


In [ ]:
soft_blend = blend_models(estimator_list=tuned_models[0:5]+[tuned_models[6]],optimize="AUC", method='soft')


In [ ]:
import os
from pycaret.classification import save_model

all_models_to_test = tuned_models + [soft_blend]
folder_name = str(MODEL_DIR)
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"folder '{folder_name}' created")

for model in all_models_to_test:
    model_name_str = str(model).split('(')[0]
    
    if model == soft_blend:
        file_name = "final_soft_blend_model"
    else:
        file_name = f"tuned_{model_name_str}"
    
    save_path = os.path.join(folder_name, file_name)
    
    save_model(model, save_path)

print(f"\n Model saved: {os.path.abspath(folder_name)}")


In [ ]:
from pycaret.classification import load_model

folder_path = str(MODEL_DIR)

model_files = [f.replace('.pkl', '') for f in os.listdir(folder_path) if f.endswith('.pkl')]

all_models_to_test = []
loaded_soft_blend = None

for model_name in model_files:
    full_path = os.path.join(folder_path, model_name)
    print(f"Loading: {model_name}...")
    
    model = load_model(full_path)
    
    if "final_soft_blend" in model_name:
        loaded_soft_blend = model
    else:
        all_models_to_test.append(model)

all_models_to_test = all_models_to_test + ([loaded_soft_blend] if loaded_soft_blend else [])

print(f"\n Loaded {len(all_models_to_test)} models.")


In [ ]:
num_cols = elec_df.columns[[43, 46]].tolist()
cat_cols = [c for c in elec_df.columns if c not in num_cols]

elec_df_train_cat = elec_df[cat_cols].copy()
elec_df_train_cat['ClassificationOfAngina_PreOp_16'] = elec_df_train_cat['ClassificationOfAngina_PreOp_16'].fillna("None")
X_train_cat = pd.get_dummies(elec_df_train_cat, columns=cat_cols)

scaler = StandardScaler()
X_train_num = pd.DataFrame(
    scaler.fit_transform(elec_df[num_cols]),
    columns=num_cols,
    index=elec_df.index
)

X_train = pd.concat([X_train_cat, X_train_num], axis=1)
train_feature_cols = X_train.columns

validate_df = pd.read_csv(DATA_DIR / "elec_2224_df.csv")
validate_y  = pd.read_csv(DATA_DIR / "elec_2224_y.csv")

val_cat = validate_df[cat_cols].copy()
val_cat['ClassificationOfAngina_PreOp_16'] = val_cat['ClassificationOfAngina_PreOp_16'].fillna("None")
X_val_cat = pd.get_dummies(val_cat, columns=cat_cols)

X_val_num = pd.DataFrame(
    scaler.transform(validate_df[num_cols]),
    columns=num_cols,
    index=validate_df.index
)

X_val = pd.concat([X_val_cat, X_val_num], axis=1)

X_val = X_val.reindex(columns=train_feature_cols, fill_value=0)

y_val = validate_y['Deathwithin30DaysSinceOT_395'].map({'Y': 1, 'N': 0}).astype(int)

validate_df_new = X_val.copy()
validate_df_new['30Ddeath'] = y_val.values


In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import VotingClassifier
results_with_ci = []

def get_model_metrics_with_ci(model, data, target_col, T_or_V, n_bootstraps=1000, alpha=0.95):

    pred_df = predict_model(model, data=data, raw_score=True)
    if isinstance(model, VotingClassifier):
        model_name = "Soft_Blend"
    else:
        if hasattr(model, 'steps'):
            model_name = type(model.steps[-1][1]).__name__
        else:
            model_name = type(model).__name__
            
    roc_df = pd.DataFrame({
        "y_true": pred_df['30Ddeath'],
        "y_score": pred_df['prediction_score_1']
    })

    file_name = f"roc_data_{model_name}_{T_or_V}.csv"
    save_path = os.path.join(folder_path, file_name)
    
    roc_df.to_csv(save_path, index=False)
    score_col = 'prediction_score_1' if 'prediction_score_1' in pred_df.columns else 'prediction_score'
    
    y_true = pred_df[target_col].values
    y_scores = pred_df[score_col].values
    
    auc_point = roc_auc_score(y_true, y_scores)
    
    bootstrapped_scores = []
    rng = np.random.RandomState(42) 
    
    for i in range(n_bootstraps):

        indices = rng.randint(0, len(y_true), len(y_true))
        
        if len(np.unique(y_true[indices])) < 2:
            continue
            
        score = roc_auc_score(y_true[indices], y_scores[indices])
        bootstrapped_scores.append(score)
    

    lower = np.percentile(bootstrapped_scores, ((1.0 - alpha) / 2.0) * 100)
    upper = np.percentile(bootstrapped_scores, (alpha + ((1.0 - alpha) / 2.0)) * 100)
    
    return {
        'Model': str(model_name),
        'AUC': round(auc_point, 4),
        '95% CI': f"({lower:.4f} - {upper:.4f})"
    }


final_results = []
for m in all_models_to_test:
    res = get_model_metrics_with_ci(m, data=validate_df_new, target_col='30Ddeath', T_or_V = 'V')
    final_results.append(res)


report_df = pd.DataFrame(final_results)
print(report_df)


In [ ]:
train_data = get_config('train')
final_results_t = []
for m in all_models_to_test:
    res = get_model_metrics_with_ci(m, data=train_data, target_col='30Ddeath', T_or_V = 'T')
    final_results_t.append(res)


report_df = pd.DataFrame(final_results_t)
print(report_df)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

roc_files = {
    "Gradient Boosting": os.path.join(folder_path, 'roc_data_GradientBoostingClassifier_V.csv'),
    "Logistic Regression": os.path.join(folder_path, 'roc_data_LogisticRegression_V.csv'),
    "Ada Boost": os.path.join(folder_path, 'roc_data_AdaBoostClassifier_V.csv'),
    "Linear DiscriminantAnalysis": os.path.join(folder_path, 'roc_data_LinearDiscriminantAnalysis_V.csv'),
    "Random Forest": os.path.join(folder_path, 'roc_data_RandomForestClassifier_V.csv'),
    "ExtraTrees": os.path.join(folder_path, 'roc_data_ExtraTreesClassifier_V.csv'),
    "Naive Bayes": os.path.join(folder_path, 'roc_data_GaussianNB_V.csv'),
    "Blend Model": os.path.join(folder_path, 'roc_data_VotingClassifier_V.csv')
}

colors = {
    "Gradient Boosting": '#0072B2',
    "Logistic Regression": '#D55E00',
    "Ada Boost": '#009E73',
    "Linear DiscriminantAnalysis": '#E69F00',
    "Random Forest": '#56B4E9',
    "ExtraTrees": '#CC79A7',
    "Naive Bayes": '#F0E442',
    "Blend Model": '#000000'
}


plt.figure(figsize=(8, 8))

for label, file in roc_files.items():
    df = pd.read_csv(file)

    y_true = df["y_true"]
    y_score = df["y_score"]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        lw=2,
        color=colors[label],
        label=f"{label} (AUC = {roc_auc:.4f})"
    )


plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.01])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation performance for 30-day mortality in elective surgery")

plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)

plt.savefig(FIGURE_DIR / "30D_elec_V.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
roc_files = {
    "Gradient Boosting": os.path.join(folder_path, 'roc_data_GradientBoostingClassifier_T.csv'),
    "Logistic Regression": os.path.join(folder_path, 'roc_data_LogisticRegression_T.csv'),
    "Ada Boost": os.path.join(folder_path, 'roc_data_AdaBoostClassifier_T.csv'),
    "Linear DiscriminantAnalysis": os.path.join(folder_path, 'roc_data_LinearDiscriminantAnalysis_T.csv'),
    "Random Forest": os.path.join(folder_path, 'roc_data_RandomForestClassifier_T.csv'),
    "ExtraTrees": os.path.join(folder_path, 'roc_data_ExtraTreesClassifier_T.csv'),
    "Naive Bayes": os.path.join(folder_path, 'roc_data_GaussianNB_T.csv'),
    "Blend Model": os.path.join(folder_path, 'roc_data_VotingClassifier_T.csv')
}

colors = {
    "Gradient Boosting": '#0072B2',
    "Logistic Regression": '#D55E00',
    "Ada Boost": '#009E73',
    "Linear DiscriminantAnalysis": '#E69F00',
    "Random Forest": '#56B4E9',
    "ExtraTrees": '#CC79A7',
    "Naive Bayes": '#F0E442',
    "Blend Model": '#000000'
}


plt.figure(figsize=(8, 8))

for label, file in roc_files.items():
    df = pd.read_csv(file)

    y_true = df["y_true"]
    y_score = df["y_score"]

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        lw=2,
        color=colors[label],
        label=f"{label} (AUC = {roc_auc:.4f})"
    )


plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.01])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Training performance for 30-day mortality in elective surgery")

plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)

plt.savefig(FIGURE_DIR / "30D_elec_T.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()


In [ ]:
best_pipe = all_models_to_test[7]
best_base_model = best_pipe.named_steps["trained_model"]
print(best_base_model)


In [ ]:
import shap
from pycaret.classification import get_config

feature_rename_map = {
    "albcont": "Serum albumin",
    "urea_ind_H": "Elevated serum urea",
    "urea_ind_N":"Normal serum urea",
    "creat_ind_H": "Elevated serum creatinine",
    "HCO3_ind_L": "Low serum bicarbonate",
    "pulse_ind_H": "Tachycardia",
    "MaxComplexityScore_PPxOthPx1_391": "Surgical complexity score",
    "FunctionalHealthStatus_PreOp_39_Independent": "Independent functional status",
    "DyspneaPriorToSurgery_PreOp2_51_Dyspnoea At Rest": "Dyspnoea at rest",
    "ASAStatus_PreOp2_38_4": "ASA physical status IV",
    "ASAStatus_PreOp2_38_3": "ASA physical status III",
    "ASAStatus_PreOp2_38_2": "ASA physical status II",
    "ASAStatus_PreOp2_38_1": "ASA physical status I",
    "PresenceOfDisseminatedCancer_26_Yes": "Disseminated cancer",
    "PresenceOfDisseminatedCancer_26_No": "No Disseminated cancer",
    "DyspneaPriorToSurgery_PreOp2_51_No Dyspnoea": "No dyspnoea prior to surgery",
    "ProcedurewithAnastomosis_412_N": "No anastomosis in surgery",
    "NeurologicalStatus_PreOp2_53_Conscious And Alert":"Conscious and alert neurological status",
    "PresenceOfSepticShockBeforeS_69_Yes":"Preoperative septic shock",
    "creat_ind_N":"Normal serum creatinine",
    "OT_Elapse_0-300":"Operative time 0–300 minutes",
    "bloodloss_ind_<100 ml":"Intraoperative blood loss < 100 mL",
    "bloodloss_ind_100-1000 ml":"Intraoperative blood loss 100–1000 mL",
    "GeneralSurgeryVascularUrolog_408_Vascular":"Vascular surgery",
    "GeneralSurgeryVascularUrolog_408_Urology":"Urological surgery",
    "sodium_ind_L":"Low serum sodium (hyponatremia)",
    "Alk_ind_H":"Elevated alkaline phosphatase"
}

X_train = get_config('X_train_transformed')

X_train = (
    X_train
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .astype(np.float64)
)

X_val = (
    X_val[X_train.columns] 
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
    .astype(np.float64)
)

X_bg  = shap.sample(X_train, 500, random_state=123)
X_exp = shap.sample(X_val, 1000, random_state=123)


try:
    explainer = shap.TreeExplainer(
        best_base_model,
        data=X_bg,
        feature_perturbation="interventional"
    )
    shap_values = explainer(X_exp)

except Exception:
    def f(X):
        return best_base_model.predict_proba(X)[:, 1]

    explainer = shap.PermutationExplainer(
        f,
        X_bg.to_numpy()
    )
    shap_values = explainer(X_exp.to_numpy())


    
X_exp = X_exp.rename(columns=feature_rename_map)
shap_values.feature_names = list(X_exp.columns)

plt.figure()
shap.summary_plot(
    shap_values,
    X_exp,
    plot_type="bar",
    max_display=12,
    show=False
)
plt.savefig(FIGURE_DIR / "figure26.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()

plt.figure()
shap.summary_plot(
    shap_values,
    X_exp,
    max_display=12,
    show=False
)

plt.savefig(FIGURE_DIR / "figure16.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close()
